# Fire Door Detection — Pipeline Testing (Colab, GPU)

Runs the full backend pipeline (ingest -> detect -> schedule parse -> cross-reference -> annotate/CSV) here instead of on a local CPU, using Colab's free T4 GPU for a large speedup on YOLO detection and OCR.

**This notebook is for development/testing only.** Production runs CPU-only by design (see `docs/AWS_COST.md`) — GPU is switched on explicitly below via `device="cuda"` / `gpu=True` params that don't exist in the deployed default path.

Before running: **Runtime -> Change runtime type -> T4 GPU**.

## 1. Clone the repo and install dependencies

In [ ]:
!git clone https://github.com/pnkkumar123/ocr-project.git
%cd ocr-project/backend

In [ ]:
# Colab already ships a CUDA-enabled torch, so only install the rest.
!pip install -q fastapi uvicorn pymupdf pydantic pydantic-settings pillow python-multipart
!pip install -q ultralytics easyocr torchvision

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only — check Runtime > Change runtime type")

## 2. Get the trained weights
`backend/models/*.pt` is gitignored (weights aren't committed), so bring `door_yolo11.pt` in separately — from the training notebook's output, or upload it here.

In [ ]:
from google.colab import files
import os

os.makedirs("models", exist_ok=True)
print("Upload door_yolo11.pt (from colab/train_fire_door_models.ipynb output):")
uploaded = files.upload()
for name in uploaded:
    os.rename(name, f"models/{name}")
print(os.listdir("models"))

## 3. Get test PDFs
`datasets/pdfs/` is also gitignored (real client-adjacent drawing sets shouldn't sit in git history). Upload the ones you want to test, or mount Drive if you keep a copy there.

In [ ]:
os.makedirs("../datasets/pdfs", exist_ok=True)
print("Upload one or more test PDFs:")
uploaded = files.upload()
for name in uploaded:
    os.rename(name, f"../datasets/pdfs/{name}")
print(os.listdir("../datasets/pdfs"))

_Alternative — mount Drive instead of re-uploading every session:_

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# PDF_DIR = '/content/drive/MyDrive/fire-door-pdfs'  # adjust to wherever you keep them

## 4. Run the full pipeline on GPU
Same `run_analysis()` the API calls, just with `DoorDetector(device="cuda")` and `OcrEngine(gpu=True)` instead of the CPU-only production defaults.

In [ ]:
import sys, time
sys.path.insert(0, ".")

from app.services.pdf_ingest import ingest_pdf
from app.services import analysis as analysis_module
from app.pipeline.door_detector import DoorDetector
from app.pipeline.ocr import OcrEngine

# Override the module's cached CPU-only singletons with GPU-enabled ones.
analysis_module._detector.cache_clear()
analysis_module._ocr_engine.cache_clear()
analysis_module._detector = lambda: DoorDetector(device="cuda")
analysis_module._ocr_engine = lambda: OcrEngine(gpu=True)

In [ ]:
PDF_PATH = "../datasets/pdfs/lumberone_arch_full_set.pdf"  # change to the file you uploaded

from pathlib import Path
data = Path(PDF_PATH).read_bytes()

t0 = time.time()
info = ingest_pdf(data, Path(PDF_PATH).name)
print(f"ingested {info.document_id[:8]} — {info.page_count} pages ({time.time()-t0:.1f}s)")

t0 = time.time()
summary = analysis_module.run_analysis(info.document_id)
print(f"analysis done in {time.time()-t0:.1f}s")
print(
    f"doors: {summary.total_doors} total | {summary.high_confidence} high-confidence | "
    f"{summary.needs_review} need review | {summary.fire_rated_located} fire-rated located on plan"
)
print(f"schedule fire-door inventory ({len(summary.schedule_fire_doors)} doors):")
for fd in summary.schedule_fire_doors:
    where = f"page {fd.located_on_page}" if fd.located_on_page is not None else "not located on plan"
    print(f"  {fd.tag}: {fd.fire_rating}  ({where})")

## 5. View an annotated page
Red = fire-rated, green = high-confidence, amber = needs review.

In [ ]:
from app.core.config import settings
from IPython.display import Image as IPyImage, display

ranked = sorted(summary.pages, key=lambda p: -len(p.doors))
top = ranked[0]
print(f"page {top.page_index}: {len(top.doors)} doors")
display(IPyImage(filename=str(settings.storage_dir / top.image_path)))

## 6. Export CSV

In [ ]:
csv_text = analysis_module.export_csv(info.document_id)
print(csv_text[:1000])

with open("export.csv", "w") as f:
    f.write(csv_text)
files.download("export.csv")

## 7. Timing comparison (optional)
Re-run step 4's `run_analysis` after switching back to CPU-only singletons to see the actual GPU speedup on this document — useful evidence for the client proposal's performance section.

In [ ]:
analysis_module._detector = lambda: DoorDetector(device="cpu")
analysis_module._ocr_engine = lambda: OcrEngine(gpu=False)

t0 = time.time()
_ = analysis_module.run_analysis(info.document_id)
print(f"CPU analysis: {time.time()-t0:.1f}s (compare to the GPU run above)")